# DockM8 v1.1: Decoy-Based Benchmarking Workflow

This notebook benchmarks DockM8 scoring workflows using known active compounds and computationally generated decoys.

**Two-phase workflow:**
1. **Optimization phase**: Generate decoys, dock them, evaluate all scoring/consensus combinations, identify the best configuration
2. **Screening phase**: Apply optimal settings to the actual compound library

**Use case:** Identify which combination of pose selection, scoring functions, and consensus method works best for your target before running a real virtual screen.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scripts.consensus.consensus import apply_consensus_scoring
from scripts.docking.docking import DOCKING_PROGRAMS, dockm8_docking
from scripts.library_preparation.library_preparation import prepare_library
from scripts.performance.analyzer import run_consensus_analysis
from scripts.pocket_finding.pocket_finder import find_pocket
from scripts.pose_selection.pose_selection import select_poses
from scripts.pose_selection.posebusters import bust_poses
from scripts.protein_preparation.protein_preparation import prepare_protein
from scripts.rescoring.rescoring import RESCORING_FUNCTIONS, rescore_poses
from scripts.utilities.fast_sdf_loader import fast_load_sdf
from scripts.utilities.fast_sdf_writer import fast_write_sdf
from software.DeepCoy.generate_decoys import generate_decoys

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

### Benchmarking Parameters
- **actives_file**: SDF file with known active compounds for decoy generation
- **n_decoys_per_active**: Decoys to generate per active (default: 10)
- **decoy_model**: DeepCoy model: `'DUDE'`, `'DEKOIS'`, or `'DUDE_P'`
- **thresholds**: Enrichment factor thresholds in percent (e.g. `[1, 2, 5]`)

### Scoring & Selection
- **pose_selection_methods**: List of methods to benchmark (e.g. `['bestpose', 'GNINA-Affinity']`)
- **rescoring_functions**: Scoring functions to evaluate in all combinations

### Other Parameters
See `dockm8.ipynb` for documentation of all standard parameters.

In [ ]:
CWD = Path(os.getcwd())
software = CWD / "software"

receptor = CWD / "test_data" / "4kd1_p.pdb"
ref_ligand = CWD / "test_data" / "4kd1_l.sdf"
docking_library = CWD / "test_data" / "library.sdf"
actives_file = CWD / "test_data" / "CDK2_actives.sdf"

pocket_mode = "Reference"
protonation = "GypsumDL"
conformers = "GypsumDL"

docking_programs = ["PLANTS", "QVINA2", "QVINAW"]
n_poses = 10
exhaustiveness = 8

pose_selection_methods = ["bestpose", "GNINA-Affinity"]

rescoring_functions = [
    "GNINA-Affinity", "CNN-Score", "CNN-Affinity", "Vinardo",
    "AD4", "KORP-PL", "ConvexPLR", "LinF9", "RTMScore", "RFScoreVS",
]

n_decoys_per_active = 10
decoy_model = "DUDE"
thresholds = [1, 2, 5]

n_cpus = max(1, int(os.cpu_count() * 0.9))

w_dir = CWD / "test_data" / "4kd1_p"
w_dir.mkdir(exist_ok=True)

print(f"Working directory: {w_dir}")
print(f"Using {n_cpus} CPUs")

## Phase 1: Decoy Optimization

### Step 1: Protein Preparation

In [ ]:
prepared_receptor = prepare_protein(
    protein_file_or_code=receptor,
    output_dir=w_dir,
    protonation_method="protoss",
)
print(f"Prepared receptor: {prepared_receptor}")

### Step 2: Pocket Identification

In [ ]:
pocket_definition = find_pocket(
    mode=pocket_mode,
    receptor=prepared_receptor,
    ligand=ref_ligand,
    radius=10,
)
print(f"Pocket: {pocket_definition}")

### Step 3: Generate Decoys

Generate computationally matched decoys for each known active compound using DeepCoy. The output file contains both actives (`Activity=1`) and decoys (`Activity=0`).

In [ ]:
decoy_dir = actives_file.parent / "DeepCoy"
decoy_dir.mkdir(exist_ok=True)

test_set_path = decoy_dir / "test_set.sdf"
if not test_set_path.exists():
    test_set_path = generate_decoys(actives_file, n_decoys_per_active, decoy_model, software)
print(f"Test set (actives + decoys): {test_set_path}")

### Step 4: Prepare the Decoy Library

In [ ]:
decoy_library_sdf = decoy_dir / "final_library.sdf"

if not decoy_library_sdf.exists():
    prepare_library(
        input_data=test_set_path,
        protonation=protonation,
        conformers=conformers,
        software=software,
        n_cpus=n_cpus,
        output_sdf=decoy_library_sdf,
    )
    print(f"Prepared decoy library: {decoy_library_sdf}")
else:
    print(f"Using existing: {decoy_library_sdf}")

### Step 5: Dock the Decoy Library

In [ ]:
all_poses_path = dockm8_docking(
    library=decoy_library_sdf,
    w_dir=decoy_dir,
    protein_file=prepared_receptor,
    pocket_definition=pocket_definition,
    software=software,
    docking_programs=docking_programs,
    exhaustiveness=exhaustiveness,
    n_poses=n_poses,
    n_cpus=n_cpus,
)
print(f"All poses: {all_poses_path}")

### Step 6: Pose Selection (Multiple Methods)

Run pose selection with each method to benchmark them all.

In [ ]:
selected_poses_dict = {}

for method in pose_selection_methods:
    output_file = decoy_dir / f"{method}_selected.sdf"
    selected = select_poses(
        selection_method=method,
        poses_input=all_poses_path,
        protein_file=prepared_receptor,
        pocket_definition=pocket_definition,
        software=software,
        output_file=output_file,
        n_cpus=n_cpus,
    )
    selected_poses_dict[method] = output_file
    print(f"  {method}: {len(selected)} poses selected")

### Step 7: Rescore All Pose Selections

In [ ]:
rescored_dict = {}

for method, poses_path in selected_poses_dict.items():
    output_file = decoy_dir / f"{method}_rescored.sdf"
    rescored = rescore_poses(
        protein_file=prepared_receptor,
        pocket_definition=pocket_definition,
        software=software,
        poses=poses_path,
        functions=rescoring_functions,
        n_cpus=n_cpus,
        output_file=output_file,
    )
    rescored_dict[method] = output_file
    print(f"  {method}: rescored {len(rescored)} poses")

### Step 8: Create Activity Data

Extract the Activity labels from the decoy library to create the ground-truth file for performance evaluation.

In [ ]:
performance_dir = decoy_dir / "performance"
performance_dir.mkdir(exist_ok=True)

activity_data_path = performance_dir / "activity_data.csv"
if not activity_data_path.exists():
    activity_df = fast_load_sdf(
        test_set_path,
        molColName=None,
        idName="ID",
        n_cpus=n_cpus,
    )
    activity_df[["ID", "Activity"]].to_csv(activity_data_path, index=False)

activity_check = pd.read_csv(activity_data_path)
print(f"Activity data: {activity_data_path}")
print(f"  Actives: {(activity_check['Activity'] == 1).sum()}, Decoys: {(activity_check['Activity'] == 0).sum()}")

### Step 9: Performance Analysis

Evaluate all combinations of scoring functions and consensus methods. The analyzer tests every 2+ function combination with each consensus method and reports enrichment factors at specified thresholds.

In [ ]:
all_performance = []

for method, rescored_path in rescored_dict.items():
    rescored_df = fast_load_sdf(rescored_path, molColName=None, idName="Pose ID", n_cpus=n_cpus)
    rescored_df["ID"] = rescored_df["Pose ID"].str.split("_").str[0]
    score_cols = [c for c in rescored_df.columns if c in RESCORING_FUNCTIONS]
    scoring_csv = performance_dir / f"{method}_scores.csv"
    rescored_df[["ID"] + score_cols].to_csv(scoring_csv, index=False)

    results = run_consensus_analysis(
        scoring_data_path=scoring_csv,
        activity_data_path=activity_data_path,
        output_path=performance_dir / f"{method}_performance.csv",
        thresholds=thresholds,
        n_jobs=n_cpus,
        include_single=True,
    )
    results["pose_selection"] = method
    all_performance.append(results)
    print(f"  {method}: {len(results)} combinations evaluated")

performance_df = pd.concat(all_performance, ignore_index=True)
print(f"\nTotal combinations evaluated: {len(performance_df)}")

### Step 10: Identify Optimal Configuration

Find the best-performing combination of pose selection, scoring functions, and consensus method based on enrichment factor at 1%.

In [ ]:
results_at_1pct = performance_df[performance_df["threshold"] == thresholds[0]]
ef_col = "ef" if "ef" in results_at_1pct.columns else results_at_1pct.columns[-1]
optimal = results_at_1pct.sort_values(ef_col, ascending=False).iloc[0]

print("Optimal configuration:")
print(f"  Pose selection: {optimal.get('pose_selection', 'N/A')}")

if "scoring_function" in optimal and pd.notna(optimal["scoring_function"]):
    optimal_functions = [optimal["scoring_function"]]
    optimal_consensus = "none"
    print(f"  Best single function: {optimal['scoring_function']}")
elif "combination" in optimal and pd.notna(optimal["combination"]):
    optimal_functions = optimal["combination"].split("+")
    optimal_consensus = optimal["consensus_method"]
    print(f"  Best combination: {optimal['combination']}")
    print(f"  Consensus method: {optimal_consensus}")
else:
    optimal_functions = rescoring_functions[:2]
    optimal_consensus = "ecr"
    print("  Using default configuration")

print(f"  Enrichment factor at {thresholds[0]}%: {optimal[ef_col]:.2f}")

## Phase 2: Virtual Screening with Optimal Settings

Apply the best configuration found above to screen the actual compound library.

### Step 11: Prepare and Dock the Screening Library

In [ ]:
screen_dir = w_dir / "screening"
screen_dir.mkdir(exist_ok=True)

screen_library_sdf = screen_dir / "final_library.sdf"
if not screen_library_sdf.exists():
    prepare_library(
        input_data=docking_library,
        protonation=protonation,
        conformers=conformers,
        software=software,
        n_cpus=n_cpus,
        output_sdf=screen_library_sdf,
    )
print(f"Screening library: {screen_library_sdf}")

screen_poses_path = dockm8_docking(
    library=screen_library_sdf,
    w_dir=screen_dir,
    protein_file=prepared_receptor,
    pocket_definition=pocket_definition,
    software=software,
    docking_programs=docking_programs,
    exhaustiveness=exhaustiveness,
    n_poses=n_poses,
    n_cpus=n_cpus,
)
print(f"Screening poses: {screen_poses_path}")

### Step 12: Select, Rescore, and Rank with Optimal Settings

In [ ]:
optimal_selection = str(optimal.get("pose_selection", "bestpose"))

selected = select_poses(
    selection_method=optimal_selection,
    poses_input=screen_poses_path,
    protein_file=prepared_receptor,
    pocket_definition=pocket_definition,
    software=software,
    output_file=screen_dir / "selected.sdf",
    n_cpus=n_cpus,
)
print(f"Selected {len(selected)} poses")

rescored = rescore_poses(
    protein_file=prepared_receptor,
    pocket_definition=pocket_definition,
    software=software,
    poses=screen_dir / "selected.sdf",
    functions=optimal_functions,
    n_cpus=n_cpus,
    output_file=screen_dir / "rescored.sdf",
)
print(f"Rescored {len(rescored)} poses")

if optimal_consensus != "none" and len(optimal_functions) > 1:
    score_cols = [c for c in rescored.columns if c in RESCORING_FUNCTIONS]
    rescored[["ID"] + score_cols].to_csv(screen_dir / "scores.csv", index=False)

    apply_consensus_scoring(
        data=screen_dir / "scores.csv",
        method=optimal_consensus,
        columns=score_cols,
        id_column="ID",
        aggregation="best",
        output=screen_dir / "final_ranking.csv",
        normalize=True,
    )
    final_df = pd.read_csv(screen_dir / "final_ranking.csv")
    print(f"\nFinal ranking ({len(final_df)} compounds):")
    print(final_df.sort_values(final_df.columns[-1], ascending=False).head(10))
else:
    print(f"\nSingle function ranking:")
    print(rescored[["ID"] + optimal_functions].sort_values(optimal_functions[0], ascending=RESCORING_FUNCTIONS[optimal_functions[0]]["best_value"] == "min").head(10))